# 00 — ISIC 2019 Dataset and Split Audit

This notebook performs **pre-model data checks** for the eight known ISIC 2019 diagnostic classes. It describes class distributions, checks image-ID exclusivity across the supplied train/validation/test splits, summarizes metadata availability, and records a reproducible split manifest.

It does **not** train models, select hyperparameters, optimize decision rules, or report predictive performance. The test split is inspected only for dataset composition and split integrity in this audit; no model predictions are generated.

## Questions addressed

1. How are the eight known diagnostic classes distributed overall and within each supplied split?
2. Are image IDs unique within each split and disjoint across train, validation, and test?
3. What proportion of split images can be matched to official ground-truth and metadata records?
4. How are age groups and anatomical regions represented, including missing values?
5. Are explicit lesion/patient identifiers available to audit entity-level separation?

## Data and privacy

Raw images and project-specific split CSVs are not included in this repository. Official label/metadata CSVs can be downloaded from their source if they are not already present. Configure paths in the next cell or use environment variables. Keep image data, derived feature arrays, model checkpoints, credentials, and private storage paths out of Git.

**Interpretation note:** Disjoint image IDs establish image-level exclusivity only. They do not establish patient- or lesion-level independence unless valid patient/lesion identifiers are available and checked.

## 1. Configure paths and load official CSV sources

Expected project layout (paths are configurable):

```text
skin_lesion_project/
├── data/
│   ├── ISIC_2019_Training_GroundTruth.csv
│   └── ISIC_2019_Training_Metadata.csv
└── splits/
    ├── train_split.csv
    ├── val_split.csv
    └── test_split.csv
```

Each split CSV must contain an image identifier column (for example `image_id`, `image`, or `filename`). If your files are elsewhere, set `SKIN_LESION_PROJECT_ROOT`, `ISIC_DATA_DIR`, and/or `ISIC_SPLIT_DIR` before running this notebook.

In [ ]:
import os
import json
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
DATA_DIR = Path(os.environ.get("ISIC_DATA_DIR", str(PROJECT_ROOT / "data")))
SPLIT_DIR = Path(os.environ.get("ISIC_SPLIT_DIR", str(PROJECT_ROOT / "splits")))
OUTPUT_DIR = Path(os.environ.get("ISIC_AUDIT_OUTPUT_DIR", str(PROJECT_ROOT / "outputs" / "dataset_audit")))
FIGURE_DIR = OUTPUT_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

GROUND_TRUTH_URL = "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv"
METADATA_URL = "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv"
GROUND_TRUTH_PATH = DATA_DIR / "ISIC_2019_Training_GroundTruth.csv"
METADATA_PATH = DATA_DIR / "ISIC_2019_Training_Metadata.csv"

SPLIT_FILES = {
    "Train": SPLIT_DIR / "train_split.csv",
    "Validation": SPLIT_DIR / "val_split.csv",
    "Test": SPLIT_DIR / "test_split.csv",
}

CLASS_ORDER = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]
AGE_EDGES = [0, 10, 20, 30, 40, 50, 60, 70, 80, np.inf]
AGE_LABELS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
FIG_DPI = 300


def download_if_missing(url: str, path: Path) -> None:
    """Download a public CSV only when the local copy does not exist."""
    if path.exists():
        print(f"Using existing file: {path.name}")
        return
    print(f"Downloading {path.name} from the official ISIC source...")
    try:
        urllib.request.urlretrieve(url, path)
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {path.name}. Place it in {DATA_DIR} and rerun. Details: {exc}"
        ) from exc
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"The download did not produce a valid file: {path}")


def normalize_image_ids(values) -> pd.Series:
    """Normalize image identifiers without changing their case."""
    s = pd.Series(values, dtype="string").str.strip()
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True, case=False)
    return s


def find_column(df: pd.DataFrame, candidates):
    """Return the first matching column, case-insensitively."""
    lookup = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


download_if_missing(GROUND_TRUTH_URL, GROUND_TRUTH_PATH)
download_if_missing(METADATA_URL, METADATA_PATH)

missing_splits = [str(path) for path in SPLIT_FILES.values() if not path.exists()]
if missing_splits:
    raise FileNotFoundError(
        "Required split CSV file(s) were not found. Configure ISIC_SPLIT_DIR or place the files here:\n"
        + "\n".join(missing_splits)
    )

print("Project root:", PROJECT_ROOT)
print("Split directory:", SPLIT_DIR)
print("Audit output directory:", OUTPUT_DIR)


## 2. Standardize labels, split identifiers, and metadata

The audit uses the eight known classes listed below. Rows without exactly one recognized known-class label (including any `UNK` row in the official ground-truth file) are counted separately rather than silently assigned to a known class.

In [ ]:
CLASS_ALIASES = {
    "AKIEC": "AK",
    "ACTINIC KERATOSIS": "AK",
    "BASAL CELL CARCINOMA": "BCC",
    "BENIGN KERATOSIS": "BKL",
    "DERMATOFIBROMA": "DF",
    "MELANOMA": "MEL",
    "MELANOCYTIC NEVUS": "NV",
    "NEVUS": "NV",
    "SQUAMOUS CELL CARCINOMA": "SCC",
    "VASCULAR LESION": "VASC",
}
IMAGE_ID_CANDIDATES = ["image", "image_id", "ISIC_ID", "Image", "filename", "fname", "img_id", "name", "file", "img", "image_path"]
AGE_CANDIDATES = ["age", "Age", "age_approx", "patient_age", "age_years", "age_approximate"]
REGION_CANDIDATES = ["anatom_site_general", "anatom_site", "region", "body_site", "anatom_site_general_challenge"]
ENTITY_CANDIDATES = ["lesion_id", "lesion", "patient_id", "patient", "subject_id", "subject", "person_id"]


def standardize_ground_truth(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    id_col = find_column(raw, IMAGE_ID_CANDIDATES)
    if id_col is None:
        raise ValueError(f"No image identifier column found in {path.name}.")
    result = pd.DataFrame({"image": normalize_image_ids(raw[id_col])})

    dx_col = find_column(raw, ["dx", "class_name", "diagnosis", "label_name"])
    if dx_col is not None:
        labels = raw[dx_col].astype("string").str.strip().str.upper()
        labels = labels.map(lambda x: CLASS_ALIASES.get(x, x) if pd.notna(x) else pd.NA)
        result["class"] = labels.where(labels.isin(CLASS_ORDER), pd.NA)
    else:
        cols_by_upper = {str(c).upper(): c for c in raw.columns}
        missing_classes = [c for c in CLASS_ORDER if c not in cols_by_upper]
        if missing_classes:
            raise ValueError(
                "Could not identify a diagnosis column or all eight one-hot class columns. "
                f"Missing one-hot column(s): {missing_classes}"
            )
        onehot = raw[[cols_by_upper[c] for c in CLASS_ORDER]].apply(pd.to_numeric, errors="coerce").fillna(0).to_numpy()
        positive_count = (onehot == 1).sum(axis=1)
        class_indices = onehot.argmax(axis=1)
        labels = np.array(CLASS_ORDER, dtype=object)[class_indices]
        result["class"] = pd.Series(np.where(positive_count == 1, labels, pd.NA), dtype="string")

    if result["image"].isna().any() or result["image"].eq("").any():
        raise ValueError("Ground-truth data contains blank image identifiers.")
    duplicates = int(result["image"].duplicated().sum())
    if duplicates:
        raise ValueError(f"Ground-truth data contains {duplicates} duplicate image identifier(s).")
    return result


def read_split(path: Path):
    raw = pd.read_csv(path)
    id_col = find_column(raw, IMAGE_ID_CANDIDATES)
    if id_col is None:
        raise ValueError(f"No image identifier column found in split file {path.name}.")
    normalized = normalize_image_ids(raw[id_col])
    invalid_mask = normalized.isna() | normalized.eq("") | normalized.str.lower().isin(["nan", "none", "null"])
    invalid_count = int(invalid_mask.sum())
    normalized = normalized.loc[~invalid_mask]
    duplicate_count = int(normalized.duplicated().sum())
    unique_ids = normalized.drop_duplicates().reset_index(drop=True)
    return pd.DataFrame({"image": unique_ids}), {
        "source_rows": int(len(raw)),
        "invalid_id_rows": invalid_count,
        "duplicate_id_rows": duplicate_count,
        "unique_image_ids": int(len(unique_ids)),
        "source_file": path.name,
    }


def standardize_metadata(path: Path) -> tuple[pd.DataFrame, dict]:
    raw = pd.read_csv(path)
    id_col = find_column(raw, IMAGE_ID_CANDIDATES)
    if id_col is None:
        raise ValueError(f"No image identifier column found in {path.name}.")
    meta = pd.DataFrame({"image": normalize_image_ids(raw[id_col])})

    age_col = find_column(raw, AGE_CANDIDATES)
    region_col = find_column(raw, REGION_CANDIDATES)
    meta["age"] = pd.to_numeric(raw[age_col], errors="coerce").clip(lower=0) if age_col else np.nan
    if region_col:
        region = raw[region_col].astype("string").str.strip().str.lower()
        meta["region"] = region.mask(region.isna() | region.isin(["", "nan", "none", "na", "null"]), "Missing")
    else:
        meta["region"] = "Missing"

    entity_cols = {}
    for candidate in ENTITY_CANDIDATES:
        col = find_column(raw, [candidate])
        if col is not None:
            standardized = candidate.replace(" ", "_")
            entity_cols[standardized] = col
            values = raw[col].astype("string").str.strip()
            values = values.mask(values.isna() | values.str.lower().isin(["", "nan", "none", "na", "null"]), pd.NA)
            meta[f"entity__{standardized}"] = values

    meta = meta.dropna(subset=["image"]).drop_duplicates("image", keep="first")
    return meta, {"age_column": age_col, "region_column": region_col, "entity_columns": entity_cols, "source_rows": int(len(raw)), "unique_image_ids": int(meta["image"].nunique())}


gt = standardize_ground_truth(GROUND_TRUTH_PATH)
splits = {}
split_source_quality = {}
for split_name, split_path in SPLIT_FILES.items():
    splits[split_name], split_source_quality[split_name] = read_split(split_path)

metadata, metadata_source_info = standardize_metadata(METADATA_PATH)

print("Known-class ground-truth rows:", int(gt["class"].isin(CLASS_ORDER).sum()))
print("Ground-truth rows not mapped to one of the eight known classes:", int(gt["class"].isna().sum()))
print("Metadata rows:", metadata_source_info["source_rows"])
print("Metadata columns used:", {k: metadata_source_info[k] for k in ["age_column", "region_column"]})
print("Available explicit entity identifier columns:", metadata_source_info["entity_columns"] or "None detected")
print("Split input quality:", json.dumps(split_source_quality, indent=2))


## 3. Class distribution and imbalance summary

Counts and percentages are descriptive. Percentages for the eight known classes use the number of images with a recognized known-class label as the denominator; unknown/unmapped and unmatched image IDs are reported separately. The split distributions are not used for model selection in this notebook.

In [ ]:
# Overall known-class distribution from the official ground-truth file.
overall_counts = gt["class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
overall_n_known = int(overall_counts.sum())
overall_table = pd.DataFrame({
    "Split": "Overall_GroundTruth",
    "Class": CLASS_ORDER,
    "Count": overall_counts.to_numpy(dtype=int),
})
overall_table["Percent_of_Known_Class_Images"] = (
    overall_table["Count"] / max(overall_n_known, 1) * 100
).round(3)

split_class_rows = []
split_quality_rows = []
class_imbalance_rows = []
gt_lookup = gt.set_index("image")["class"]
gt_id_set = set(gt["image"].astype(str))

for split_name, split_df in splits.items():
    merged = split_df.merge(gt, on="image", how="left", validate="one_to_one", indicator=True)
    known = merged["class"].isin(CLASS_ORDER)
    known_counts = merged.loc[known, "class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    n_known = int(known_counts.sum())
    unmatched_ids = int((merged["_merge"] == "left_only").sum())
    matched_but_unmapped = int(((merged["_merge"] == "both") & merged["class"].isna()).sum())

    for class_name in CLASS_ORDER:
        count = int(known_counts[class_name])
        split_class_rows.append({
            "Split": split_name,
            "Class": class_name,
            "Count": count,
            "Percent_of_Known_Class_Images": round(100 * count / max(n_known, 1), 3),
        })

    positive_counts = known_counts[known_counts > 0]
    imbalance_ratio = float(positive_counts.max() / positive_counts.min()) if len(positive_counts) == len(CLASS_ORDER) else np.inf
    class_imbalance_rows.append({
        "Split": split_name,
        "Known_Class_Images": n_known,
        "Minority_Class_Count": int(known_counts.min()),
        "Majority_Class_Count": int(known_counts.max()),
        "Majority_to_Minority_Ratio": imbalance_ratio,
        "Classes_With_Zero_Count": ", ".join(known_counts[known_counts == 0].index.tolist()),
    })
    split_quality_rows.append({
        "Split": split_name,
        **split_source_quality[split_name],
        "Known_Class_Images": n_known,
        "Matched_GroundTruth_But_Unmapped": matched_but_unmapped,
        "Not_Found_In_GroundTruth": unmatched_ids,
        "GroundTruth_Match_Percent": round(100 * (len(merged) - unmatched_ids) / max(len(merged), 1), 3),
    })

split_class_table = pd.DataFrame(split_class_rows)
split_quality_table = pd.DataFrame(split_quality_rows)
class_imbalance_table = pd.DataFrame(class_imbalance_rows)

print("Overall known-class distribution:")
display(overall_table)
print("Per-split known-class distribution:")
display(split_class_table)
print("Split quality and label coverage:")
display(split_quality_table)
print("Class imbalance summary:")
display(class_imbalance_table)

# Save machine-readable outputs.
overall_table.to_csv(OUTPUT_DIR / "overall_class_distribution.csv", index=False)
split_class_table.to_csv(OUTPUT_DIR / "split_class_distribution.csv", index=False)
split_quality_table.to_csv(OUTPUT_DIR / "split_quality_summary.csv", index=False)
class_imbalance_table.to_csv(OUTPUT_DIR / "class_imbalance_summary.csv", index=False)

# Plot the split class counts for a visual check. No hypothesis test is performed here.
plot_data = split_class_table.pivot(index="Class", columns="Split", values="Count").reindex(CLASS_ORDER)
ax = plot_data.plot(kind="bar", figsize=(11, 6))
ax.set_title("Known-Class Distribution by Split")
ax.set_xlabel("Diagnostic Class")
ax.set_ylabel("Image Count")
ax.legend(title="Split")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_distribution_by_split.png", dpi=FIG_DPI, bbox_inches="tight")
plt.savefig(FIGURE_DIR / "class_distribution_by_split.pdf", bbox_inches="tight")
plt.show()
plt.close()


## 4. Image-ID exclusivity and reproducibility manifest

This section checks duplicate image IDs within each split and pairwise overlap across splits. It creates SHA-256 hashes from the sorted unique image-ID lists so that the split membership can be recorded and compared later.

**Scope limitation:** If the metadata does not provide reliable patient/lesion identifiers, image-ID exclusivity cannot prove patient- or lesion-level independence. The notebook reports that limitation rather than claiming that it has been proven.

In [ ]:
SPLIT_NAMES = list(splits.keys())
split_id_sets = {name: set(df["image"].astype(str)) for name, df in splits.items()}
overlap_matrix = pd.DataFrame(0, index=SPLIT_NAMES, columns=SPLIT_NAMES, dtype=int)
for left in SPLIT_NAMES:
    for right in SPLIT_NAMES:
        overlap_matrix.loc[left, right] = len(split_id_sets[left] & split_id_sets[right])

overlap_pairs = []
for i, left in enumerate(SPLIT_NAMES):
    for right in SPLIT_NAMES[i + 1:]:
        intersection = sorted(split_id_sets[left] & split_id_sets[right])
        overlap_pairs.append({
            "Split_A": left,
            "Split_B": right,
            "Overlap_Count": len(intersection),
            "Example_Overlapping_IDs": "; ".join(intersection[:10]),
        })
overlap_pair_table = pd.DataFrame(overlap_pairs)


def sha256_sorted_ids(ids):
    payload = "\n".join(sorted(map(str, ids))).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

manifest = {
    "audit_scope": "image-level split integrity and descriptive dataset audit",
    "class_order": CLASS_ORDER,
    "input_files": {
        "ground_truth": GROUND_TRUTH_PATH.name,
        "metadata": METADATA_PATH.name,
        "splits": {name: info["source_file"] for name, info in split_source_quality.items()},
    },
    "split_summary": {},
    "pairwise_image_id_overlap": overlap_pairs,
    "metadata_source_info": metadata_source_info,
    "entity_level_independence": (
        "Entity identifier availability and cross-split overlaps are assessed in the metadata audit section. "
        "Image-level exclusivity alone does not establish patient- or lesion-level independence."
    ),
}

for split_name, split_df in splits.items():
    ids = split_df["image"].astype(str).tolist()
    manifest["split_summary"][split_name] = {
        "unique_image_count": len(set(ids)),
        "duplicate_rows_in_source": split_source_quality[split_name]["duplicate_id_rows"],
        "invalid_id_rows_in_source": split_source_quality[split_name]["invalid_id_rows"],
        "sha256_sorted_unique_image_ids": sha256_sorted_ids(ids),
        "images_found_in_ground_truth": int(len(set(ids) & gt_id_set)),
        "images_not_found_in_ground_truth": int(len(set(ids) - gt_id_set)),
    }

overlap_matrix.to_csv(OUTPUT_DIR / "split_overlap_matrix.csv")
overlap_pair_table.to_csv(OUTPUT_DIR / "split_overlap_pairs.csv", index=False)
with open(OUTPUT_DIR / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Pairwise image-ID overlap (diagonal = split size; off-diagonal should be zero):")
display(overlap_matrix)
print("Pairwise overlap details:")
display(overlap_pair_table)
print("Split manifest:")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(overlap_matrix.to_numpy(), aspect="auto")
ax.set_xticks(range(len(SPLIT_NAMES)), SPLIT_NAMES, rotation=25, ha="right")
ax.set_yticks(range(len(SPLIT_NAMES)), SPLIT_NAMES)
ax.set_title("Pairwise Image-ID Overlap")
for i in range(len(SPLIT_NAMES)):
    for j in range(len(SPLIT_NAMES)):
        ax.text(j, i, str(int(overlap_matrix.iloc[i, j])), ha="center", va="center")
fig.colorbar(im, ax=ax, label="Number of shared image IDs")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "split_overlap_matrix.png", dpi=FIG_DPI, bbox_inches="tight")
fig.savefig(FIGURE_DIR / "split_overlap_matrix.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

if any(row["Overlap_Count"] > 0 for row in overlap_pairs):
    print("WARNING: At least one image ID occurs in more than one split. Investigate before modeling.")
else:
    print("No pairwise image-ID overlap was detected among the supplied splits.")


## 5. Metadata availability and distribution

Age and anatomical-region distributions are reported descriptively for each split. Missing metadata values are kept as explicit categories. This section does not perform significance tests or make a claim that the splits are statistically equivalent.

In [ ]:
metadata_coverage_rows = []
age_distribution_rows = []
region_distribution_rows = []
entity_overlap_rows = []
entity_sets_by_type = {}

metadata_lookup = metadata.set_index("image", drop=False)

for split_name, split_df in splits.items():
    merged = split_df.merge(metadata, on="image", how="left", validate="one_to_one", suffixes=("", "_meta"))
    n_images = len(merged)
    age_group = pd.cut(merged["age"], bins=AGE_EDGES, labels=AGE_LABELS, right=False, include_lowest=True).astype("string")
    age_group = age_group.fillna("Missing")
    region = merged["region"].astype("string").fillna("Missing")
    region = region.mask(region.str.strip().str.lower().isin(["", "nan", "none", "na", "null"]), "Missing")

    age_observed = int(merged["age"].notna().sum())
    region_observed = int(region.ne("Missing").sum())
    # Match coverage is based on whether the image ID exists in the metadata table.
    metadata_matched_count = int(merged["image"].isin(set(metadata["image"])).sum())
    metadata_coverage_rows.append({
        "Split": split_name,
        "Images": n_images,
        "Images_Matched_To_Metadata": metadata_matched_count,
        "Metadata_Match_Percent": round(100 * metadata_matched_count / max(n_images, 1), 3),
        "Age_Observed": age_observed,
        "Age_Missing_or_Unavailable": n_images - age_observed,
        "Age_Missing_Percent": round(100 * (n_images - age_observed) / max(n_images, 1), 3),
        "Region_Observed": region_observed,
        "Region_Missing_or_Unavailable": n_images - region_observed,
        "Region_Missing_Percent": round(100 * (n_images - region_observed) / max(n_images, 1), 3),
    })

    age_counts = age_group.value_counts().reindex(AGE_LABELS + ["Missing"], fill_value=0)
    for category, count in age_counts.items():
        age_distribution_rows.append({
            "Split": split_name, "Age_Group": category, "Count": int(count),
            "Percent_of_Split": round(100 * int(count) / max(n_images, 1), 3),
        })

    region_counts = region.value_counts(dropna=False)
    for category, count in region_counts.items():
        region_distribution_rows.append({
            "Split": split_name, "Region": str(category), "Count": int(count),
            "Percent_of_Split": round(100 * int(count) / max(n_images, 1), 3),
        })

    for entity_name in metadata_source_info["entity_columns"]:
        entity_col = f"entity__{entity_name}"
        if entity_col not in merged.columns:
            continue
        vals = merged[entity_col].dropna().astype(str)
        vals = vals[~vals.str.lower().isin(["", "nan", "none", "na", "null"])]
        entity_overlap_rows.append({
            "Split": split_name,
            "Entity_ID_Type": entity_name,
            "Unique_Nonmissing_Entity_IDs": int(vals.nunique()),
            "Images_With_Entity_ID": int(len(vals)),
            "Images_Missing_Entity_ID": int(n_images - len(vals)),
        })
        entity_sets_by_type.setdefault(entity_name, {})[split_name] = set(vals.tolist())

metadata_coverage = pd.DataFrame(metadata_coverage_rows)
age_distribution = pd.DataFrame(age_distribution_rows)
region_distribution = pd.DataFrame(region_distribution_rows)
entity_availability = pd.DataFrame(entity_overlap_rows)
entity_pair_rows = []
for entity_name, sets_by_split in entity_sets_by_type.items():
    for i, left in enumerate(SPLIT_NAMES):
        for right in SPLIT_NAMES[i + 1:]:
            left_ids = sets_by_split.get(left, set())
            right_ids = sets_by_split.get(right, set())
            shared = sorted(left_ids & right_ids)
            entity_pair_rows.append({
                "Entity_ID_Type": entity_name,
                "Split_A": left,
                "Split_B": right,
                "Shared_Nonmissing_Entity_IDs": len(shared),
                "Example_Shared_Entity_IDs": "; ".join(shared[:10]),
            })
entity_overlap_table = pd.DataFrame(entity_pair_rows)
if entity_availability.empty:
    entity_availability = pd.DataFrame([{
        "Status": "No explicit lesion/patient/subject identifier columns were detected in the metadata source."
    }])
if entity_overlap_table.empty:
    entity_overlap_table = pd.DataFrame([{
        "Status": "Entity-level overlap cannot be assessed because no explicit entity identifier columns were detected."
    }])

metadata_coverage.to_csv(OUTPUT_DIR / "metadata_coverage_by_split.csv", index=False)
age_distribution.to_csv(OUTPUT_DIR / "age_distribution_by_split.csv", index=False)
region_distribution.to_csv(OUTPUT_DIR / "region_distribution_by_split.csv", index=False)
entity_availability.to_csv(OUTPUT_DIR / "entity_identifier_availability.csv", index=False)
entity_overlap_table.to_csv(OUTPUT_DIR / "entity_identifier_overlap.csv", index=False)

manifest["entity_level_overlap"] = entity_pair_rows
if not metadata_source_info["entity_columns"]:
    manifest["entity_level_independence"] = (
        "Not assessable: no explicit lesion/patient/subject identifier columns were detected. "
        "Zero image-ID overlap does not establish lesion- or patient-level independence."
    )
elif any(row["Shared_Nonmissing_Entity_IDs"] > 0 for row in entity_pair_rows):
    manifest["entity_level_independence"] = (
        "At least one shared non-missing entity identifier was detected across splits. Investigate before modeling. "
        "Images with missing entity identifiers cannot be assessed at entity level."
    )
else:
    manifest["entity_level_independence"] = (
        "No shared non-missing identifiers were detected for the available entity columns. "
        "Images with missing entity identifiers cannot be assessed at entity level."
    )
with open(OUTPUT_DIR / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Metadata coverage and missingness:")
display(metadata_coverage)
print("Age-group distribution:")
display(age_distribution)
print("Anatomical-region distribution:")
display(region_distribution.sort_values(["Split", "Count"], ascending=[True, False]))
if metadata_source_info["entity_columns"]:
    print("Available entity identifier coverage:")
    display(entity_availability)
else:
    print("No explicit lesion/patient identifier columns were detected in the metadata source.")
print("Entity-level overlap among available non-missing identifiers:")
display(entity_overlap_table)


## 6. Review checklist and interpretation limits

Before modeling, review the generated CSV files and manifest. Investigate any duplicate IDs, non-zero overlaps, unmatched ground-truth IDs, or unexpected metadata missingness.

- A zero image-ID overlap is necessary for image-level split exclusivity, but it is not sufficient to establish patient/lesion-level independence.
- Distribution tables and plots are descriptive. They do not prove that the splits are statistically equivalent.
- The eight-class counts exclude rows that do not map to exactly one known class. These rows are reported separately in the split-quality summary.
- This notebook does not alter split membership, train models, tune parameters, choose a final model, or evaluate predictions.
- Do not commit raw images, private metadata, split CSVs that should remain restricted, large derived arrays, or model checkpoints unless their redistribution is permitted and intended.

## Outputs

Generated outputs are written to `OUTPUT_DIR`, including class distribution tables, metadata distribution tables, split-overlap checks, SHA-256 split manifests, and two audit figures.